# Kaggle Playground Series S6E5 — Predict F1 Pit Stops
### Author: Ruide Yin

## Stage 6: Post-processing & Submission

**Goal:** Load pipeline outputs, sanity-check predictions, and generate `submission.csv`.  
**Metric:** AUC-ROC  
**No training is performed in this notebook — load only.**

### 6.1 Imports

In [ ]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt

OUTPUT_DIR = "./outputs"

### 6.2 Load Test Data & Confirm ID Column

In [ ]:
test = pd.read_csv(f"{OUTPUT_DIR}/test_fe.csv")
print(f"Test shape: {test.shape}")
print(f"Columns ({len(test.columns)}): {list(test.columns)}")
print(f"\nID column stats:")
print(f"  dtype:  {test['id'].dtype}")
print(f"  range:  {test['id'].min()} ~ {test['id'].max()}")
print(f"  nunique: {test['id'].nunique()}  (should == {len(test)})")
assert test["id"].nunique() == len(test), "ID column has duplicates!"

test_ids = test["id"]
test_ids.head(10)

### 6.3 Load Ensemble Configuration

In [ ]:
with open(f"{OUTPUT_DIR}/ensemble_config.pkl", "rb") as f:
    ens_cfg = pickle.load(f)

print("=" * 60)
print("Pipeline Selected Ensemble")
print("=" * 60)
print(f"  Method:      {ens_cfg.get('method', 'N/A')}")
print(f"  Final AUC:   {ens_cfg.get('final_auc', 'N/A')}")
print(f"\n  Selected models:")
for i, m in enumerate(ens_cfg.get("selected", [])):
    w = ens_cfg.get("weights", [None])[i] if ens_cfg.get("weights") else None
    w_str = f"  (weight={w:.4f})" if w is not None else ""
    print(f"    [{i+1}] {m}{w_str}")
print("=" * 60)

### 6.4 Load Final Test Predictions

In [ ]:
test_final = np.load(f"{OUTPUT_DIR}/test_final.npy")
print(f"test_final shape: {test_final.shape}")
print(f"test_final dtype: {test_final.dtype}")

### 6.5 Load All Single-Model Predictions

In [ ]:
MODEL_NAMES = [
    "LGB_tuned", "XGB_tuned", "CAT_baseline",
    "RF_baseline", "MLP_tuned", "LR_baseline",
]

preds = {}
for name in MODEL_NAMES:
    path = f"{OUTPUT_DIR}/test_{name}.npy"
    try:
        preds[name] = np.load(path)
        print(f"  {name:15s}  shape={preds[name].shape}")
    except FileNotFoundError:
        print(f"  {name:15s}  NOT FOUND — skipped")

# Also load the two ensemble variants for reference
for tag in ["ensemble_wa", "ensemble_stack"]:
    path = f"{OUTPUT_DIR}/test_{tag}.npy"
    try:
        preds[tag] = np.load(path)
        print(f"  {tag:15s}  shape={preds[tag].shape}")
    except FileNotFoundError:
        print(f"  {tag:15s}  NOT FOUND — skipped")

print(f"\nLoaded {len(preds)} prediction arrays in total.")

### 6.6 Sanity Checks

#### 6.6.1 Basic Statistics & NaN / Range Check

In [ ]:
TRAIN_POS_RATE = 0.199  # from EDA: ~19.9%

print(f"{'Source':<18s} {'Mean':>8s} {'Median':>8s} {'Min':>8s} {'Max':>8s} {'Std':>8s} {'NaN':>5s} {'OOB':>5s}")
print("-" * 80)

all_arrays = {"test_final": test_final, **preds}

for name, arr in all_arrays.items():
    n_nan = int(np.isnan(arr).sum())
    n_oob = int(((arr < 0) | (arr > 1)).sum()) - n_nan  # out-of-bound excluding NaN
    print(f"{name:<18s} {arr.mean():8.5f} {np.median(arr):8.5f} {arr.min():8.5f} {arr.max():8.5f} {arr.std():8.5f} {n_nan:5d} {n_oob:5d}")

print(f"\n{'Train pos rate':<18s} {TRAIN_POS_RATE:8.5f}")
print("\n[OK] If mean ≈ 0.199, predictions are well-calibrated.")
print("[OK] NaN = 0 and OOB = 0 for all sources → safe to submit.")

#### 6.6.2 Prediction Distribution Histogram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: test_final
axes[0].hist(test_final, bins=100, edgecolor="none", alpha=0.8)
axes[0].axvline(TRAIN_POS_RATE, color="red", ls="--", label=f"Train pos rate ({TRAIN_POS_RATE})")
axes[0].axvline(test_final.mean(), color="orange", ls="--", label=f"Pred mean ({test_final.mean():.4f})")
axes[0].set_title("test_final distribution")
axes[0].set_xlabel("Predicted P(PitNextLap)")
axes[0].legend(fontsize=8)

# Right: all single models overlaid
for name, arr in preds.items():
    if name.startswith("ensemble"):
        continue
    axes[1].hist(arr, bins=100, alpha=0.4, label=name, edgecolor="none")
axes[1].axvline(TRAIN_POS_RATE, color="red", ls="--", label="Train pos rate")
axes[1].set_title("Single model distributions")
axes[1].set_xlabel("Predicted P(PitNextLap)")
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

### 6.7 Generate `submission.csv`

In [ ]:
submission = pd.DataFrame({
    "id": test_ids,
    "PitNextLap": test_final,
})

# Final assertions
assert len(submission) == len(test), f"Row count mismatch: {len(submission)} vs {len(test)}"
assert submission["PitNextLap"].isna().sum() == 0, "NaN in predictions!"
assert submission["PitNextLap"].between(0, 1).all(), "Predictions out of [0,1]!"

save_path = f"{OUTPUT_DIR}/submission.csv"
submission.to_csv(save_path, index=False)
print(f"Saved to {save_path}")
print(f"  Rows:  {len(submission)}")
print(f"  Mean:  {submission['PitNextLap'].mean():.5f}")
submission.head(10)